# VisionDrive 3D — Colab Export

Notebook ini menjalankan pipeline deteksi YOLO + visualisasi pseudo-3D secara headless (tanpa window),
lalu menyimpan hasilnya sebagai file video `.mp4`.

**Langkah:**
1. Clone repo dan install dependensi
2. Upload video input
3. Jalankan export
4. Download video output

## 1. Clone repo dan install dependensi

In [ ]:
import os

REPO_URL = "https://github.com/homomachinus/visiondrive3d.git"  # ganti dengan URL repo kamu
REPO_DIR = "visiondrive3d"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("repo sudah ada, skip clone.")

%cd {REPO_DIR}

In [ ]:
# install semua dependensi
!pip install -q ultralytics opencv-python-headless pygame numpy

## 2. Upload video input

Upload file video dashcam kamu. Default nama file yang dicari adalah `vx1.mp4`.
Kalau nama berbeda, ubah variabel `VIDEO_INPUT` di cell berikutnya.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 3. Konfigurasi export

In [ ]:
VIDEO_INPUT  = "vx1.mp4"      # nama file video yg sudah diupload
VIDEO_OUTPUT = "output.mp4"   # nama file output
MAX_FRAMES   = None           # None = proses semua frame, isi angka untuk batasi (misal: 300)
EGO_SPEED    = 10.0           # kecepatan simulasi ego vehicle (m/s)

## 4. Jalankan export headless

In [ ]:
import os
import sys

# paksa pygame render offscreen (tanpa window)
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"

import cv2
import numpy as np
import pygame
import time
from ultralytics import YOLO

import src as v


# --- konstanta warna & dimensi kendaraan ---
CLASS_COLORS = {
    "car":        (0, 200, 210),
    "motorcycle": (210, 100, 0),
    "bus":        (100, 210, 50),
    "truck":      (200, 50, 100),
}
CLASS_DIMS = {
    "motorcycle": (0.8, 2.0, 1.5),
    "bus":        (2.8, 10.0, 3.5),
    "truck":      (2.5, 8.0, 3.0),
}
DEFAULT_DIM    = (2.0, 4.5, 1.5)
TARGET_CLASSES = ["car", "motorcycle"]


def get_color(cls_name):
    return CLASS_COLORS.get(cls_name, v.C_OTHER)

def get_dimensions(cls_name):
    return CLASS_DIMS.get(cls_name, DEFAULT_DIM)

def filter_occlusion(vehicles):
    filtered = []
    for i, obj in enumerate(vehicles):
        is_occluded = False
        x1_1, y1_1, x2_1, y2_1 = obj['box']
        for j, other in enumerate(vehicles):
            if i == j:
                continue
            x1_2, y1_2, x2_2, y2_2 = other['box']
            ix1, iy1 = max(x1_1, x1_2), max(y1_1, y1_2)
            ix2, iy2 = min(x2_1, x2_2), min(y2_1, y2_2)
            if ix2 > ix1 and iy2 > iy1:
                inter = (ix2 - ix1) * (iy2 - iy1)
                a1 = max(1, (x2_1 - x1_1) * (y2_1 - y1_1))
                a2 = max(1, (x2_2 - x1_2) * (y2_2 - y1_2))
                if inter / min(a1, a2) > 0.5 and other['depth'] < obj['depth']:
                    is_occluded = True
                    break
        if not is_occluded:
            filtered.append(obj)
    return filtered


class Tracker:
    def __init__(self, alpha=0.15):
        self.history = {}
        self.alpha   = alpha

    def cleanup(self, current_ids):
        for oid in list(self.history):
            if oid not in current_ids:
                del self.history[oid]

    def get_locked_class(self, oid):
        val = self.history.get(oid)
        if val is None:
            return None
        return val['cls'] if isinstance(val, dict) else val[2]

    def smooth(self, oid, wx, depth):
        if oid in self.history and not isinstance(self.history[oid], dict):
            pw, pd, _ = self.history[oid]
            wx    = self.alpha * wx    + (1 - self.alpha) * pw
            depth = self.alpha * depth + (1 - self.alpha) * pd
        return wx, depth

    def save(self, oid, wx, depth, cls_name):
        self.history[oid] = (wx, depth, cls_name)


class HeadlessExporter:
    """sama seperti InferVisualizer tapi render ke video, bukan ke window"""

    def __init__(self, video_input, video_output, max_frames=None, ego_speed=10.0):
        self.video_output = video_output
        self.max_frames   = max_frames

        # init pygame headless
        pygame.init()
        self.screen = pygame.display.set_mode((v.W, v.H))

        self.model   = YOLO("yolo11n.pt")
        self.cap     = cv2.VideoCapture(video_input)
        self.video_w = self.cap.get(cv2.CAP_PROP_FRAME_WIDTH)
        self.video_h = self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
        self.fps_in  = self.cap.get(cv2.CAP_PROP_FPS) or 30.0

        self.ego       = v.EgoVehicle()
        self.ego.speed = ego_speed
        self.sim_t     = 0.0
        self.tick      = 0
        self.tracker   = Tracker(alpha=0.15)

        # setup video writer — codec mp4v untuk output .mp4
        fourcc   = cv2.VideoWriter_fourcc(*'mp4v')
        self.out = cv2.VideoWriter(video_output, fourcc, self.fps_in, (v.W, v.H))

    def _estimate_depth(self, bbox_h, y2, wh, focal_px, horizon_y):
        d_h = (wh * focal_px) / max(1, bbox_h)
        d_y = ((self.video_h - horizon_y) * 4.0) / max(1, y2 - horizon_y)
        return max(2.0, min(d_h * 0.7 + d_y * 0.3, 150.0))

    def _process_frame(self, frame, focal_px, horizon_y):
        results = self.model.track(frame, persist=True, verbose=False, conf=0.45)[0]
        current_ids = set()
        if results.boxes.id is not None:
            current_ids = set(results.boxes.id.cpu().numpy().astype(int))
        self.tracker.cleanup(current_ids)

        vehicles = []
        for box in results.boxes:
            cls_id   = int(box.cls[0])
            cls_name = self.model.names[cls_id]
            obj_id   = int(box.id[0]) if box.id is not None else None
            if obj_id is not None:
                locked = self.tracker.get_locked_class(obj_id)
                if locked:
                    cls_name = locked
            if cls_name not in TARGET_CLASSES:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            ww, wl, wh = get_dimensions(cls_name)
            depth = self._estimate_depth(y2 - y1, y2, wh, focal_px, horizon_y)
            cx    = (x1 + x2) / 2.0
            wx    = self.ego.x + ((cx - self.video_w / 2.0) / focal_px) * depth
            if obj_id is not None:
                wx, depth = self.tracker.smooth(obj_id, wx, depth)
                self.tracker.save(obj_id, wx, depth, cls_name)
            wy       = self.ego.y + depth
            selected = abs(wx - self.ego.x) < (v.ROAD_W / 3) * 0.8 and depth < 35.0
            if abs(wx - self.ego.x) <= v.ROAD_W / 2.0:
                vehicles.append({
                    'wx': wx, 'wy': wy, 'wz': 0.0,
                    'ww': ww, 'wl': wl, 'wh': wh,
                    'color': get_color(cls_name),
                    'selected': selected,
                    'depth': depth,
                    'box': (x1, y1, x2, y2),
                })
        return results, filter_occlusion(vehicles)

    def _surface_to_bgr(self):
        """ambil pixel dari pygame surface → numpy BGR untuk cv2"""
        # surfarray menghasilkan shape (W, H, 3), perlu transpose ke (H, W, 3)
        rgb = pygame.surfarray.array3d(self.screen).transpose(1, 0, 2)
        return cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)

    def _make_pip_surf(self, annotated_frame, vehicles, pip_w=320, pip_h=180):
        """buat surface pip dari frame yolo yang sudah dianotasi"""
        for obj in vehicles:
            x1, y1, x2, y2 = obj['box']
            cv2.putText(
                annotated_frame, f"{obj['depth']:.1f}m",
                (x1 + 5, max(30, y2 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA
            )
        small = cv2.resize(annotated_frame, (pip_w, pip_h))
        rgb   = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
        return pygame.surfarray.make_surface(np.swapaxes(rgb, 0, 1))

    def run(self):
        focal_px  = self.video_h * 1.2
        horizon_y = self.video_h * 0.45
        dt        = 1.0 / self.fps_in   # dt konstan karena tidak ada real-time

        detected_vehicles = []
        cam_surf          = None
        frame_count       = 0
        total_frames      = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        limit             = self.max_frames or total_frames

        print(f"memproses {min(limit, total_frames)} frame dari total {total_frames} frame...")

        while self.cap.isOpened() and frame_count < limit:
            ret, frame = self.cap.read()
            if not ret:
                break

            self.tick  += 1
            self.sim_t += dt
            self.ego.y += self.ego.speed * dt

            results, detected_vehicles = self._process_frame(frame, focal_px, horizon_y)
            cam_surf = self._make_pip_surf(results.plot(), detected_vehicles)

            # render scene ke pygame surface (headless)
            self.screen.fill(v.C_BG)
            cam_x = self.ego.x
            cam_y = self.ego.y + v.CAM_Y_OFFSET

            v.draw_road(self.screen, cam_x, cam_y, self.tick)

            for obj in detected_vehicles:
                v.draw_box(
                    self.screen, cam_x, cam_y,
                    obj['wx'], obj['wy'], obj['ww'], obj['wl'], obj['wh'],
                    obj['color'], selected=obj['selected'], is_ego=False, wz=obj['wz']
                )

            v.draw_box(
                self.screen, cam_x, cam_y,
                self.ego.x, self.ego.y, self.ego.w, self.ego.l, self.ego.h,
                v.C_EGO, is_ego=True
            )

            v.draw_hud(self.screen, self.ego, self.tick, self.sim_t)

            # pip di pojok kanan bawah
            if cam_surf:
                pip_w, pip_h, pad = 320, 180, 20
                pip_x = v.W - pip_w - pad
                pip_y = v.H - pip_h - pad
                pygame.draw.rect(self.screen, (200, 200, 200),
                                 (pip_x-2, pip_y-2, pip_w+4, pip_h+4), 2)
                self.screen.blit(cam_surf, (pip_x, pip_y))

            # ambil frame dari surface dan tulis ke video
            self.out.write(self._surface_to_bgr())

            frame_count += 1
            if frame_count % 30 == 0:
                pct = frame_count / min(limit, total_frames) * 100
                print(f"  {frame_count}/{min(limit, total_frames)} frame ({pct:.1f}%)")

        self.cap.release()
        self.out.release()
        pygame.quit()
        print(f"\nselesai! video disimpan ke: {self.video_output}")


# jalankan export
exporter = HeadlessExporter(
    video_input  = VIDEO_INPUT,
    video_output = VIDEO_OUTPUT,
    max_frames   = MAX_FRAMES,
    ego_speed    = EGO_SPEED,
)
exporter.run()

## 5. Download video output

In [ ]:
from google.colab import files
files.download(VIDEO_OUTPUT)

## (Opsional) Preview video langsung di notebook

In [ ]:
from IPython.display import HTML
from base64 import b64encode

with open(VIDEO_OUTPUT, 'rb') as f:
    video_b64 = b64encode(f.read()).decode()

HTML(f"""
<video width="800" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
""")